## Tracking benchmark

Compare MigFinder tracking labels against GT tracking labels for panelc videos `09`, `26`, and `32`.


Run the configuration cell first. Replace its path placeholders for the analysis steps you intend to run; later cells reuse these shared settings.


In [ ]:
from IPython.display import display

# Set paths for the analysis steps you intend to run.
# Each path is checked when a later cell first uses it.
PATHS = {'gt_root': '/path/of/annotations/tracking', 'pred_label_root': '/path/of/predictions/tracking', 'output_dir': '/path/of/results/tracking/benchmark'}

def configured_path(key):
    """Return a configured path; stop before using an unedited placeholder."""
    value = PATHS[key]
    if value.startswith('/path/of/'):
        raise ValueError(f"Replace PATHS[{key!r}] in the configuration cell with your own path.")
    return value


In [ ]:
import os
from pathlib import Path
import re

import motmetrics as mm
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.size': 14})

GT_ROOT = Path(configured_path('gt_root'))
IMAGE_ROOT = GT_ROOT / 'images'
GT_LABEL_ROOT = GT_ROOT / 'labels'
PRED_LABEL_ROOT = Path(configured_path('pred_label_root'))
OUTPUT_DIR = Path(configured_path('output_dir'))

VIDEO_FOLDERS = ['09', '26', '32']
METHOD_NAME = 'MigFinder'
IOU_THRESHOLD = 0.5
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
FRAME_RE = re.compile(r'R(\d+)$')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def frame_sort_key(path: Path):
    match = FRAME_RE.search(path.stem)
    return int(match.group(1)) if match else path.name


def list_images(video: str):
    image_dir = IMAGE_ROOT / video
    images = [p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    return sorted(images, key=frame_sort_key)


def yolo_rows_to_xyxy(rows: np.ndarray, image_width: int, image_height: int):
    ids = rows[:, 0].astype(int)
    xc = rows[:, 1] * image_width
    yc = rows[:, 2] * image_height
    bw = rows[:, 3] * image_width
    bh = rows[:, 4] * image_height
    boxes = np.column_stack([
        xc - bw / 2.0,
        yc - bh / 2.0,
        xc + bw / 2.0,
        yc + bh / 2.0,
    ])
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, image_width)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, image_height)
    return ids, boxes


def read_yolo_frame(label_path: Path, image_size: tuple[int, int], video: str):
    image_width, image_height = image_size
    if (not label_path.exists()) or label_path.stat().st_size == 0:
        return [], np.empty((0, 4), dtype=float)

    rows = np.loadtxt(label_path, ndmin=2)
    if rows.size == 0:
        return [], np.empty((0, 4), dtype=float)
    if rows.shape[1] < 5:
        raise ValueError(f'{label_path} should have at least 5 columns, got {rows.shape[1]}')

    rows = rows[:, :5]
    ids, boxes = yolo_rows_to_xyxy(rows, image_width, image_height)
    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    ids = ids[valid]
    boxes = boxes[valid]

    namespaced_ids = [f'{video}_{track_id}' for track_id in ids]
    return namespaced_ids, boxes


def bbox_iou_matrix(gt_boxes: np.ndarray, pred_boxes: np.ndarray, iou_threshold: float):
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return np.empty((len(gt_boxes), len(pred_boxes)))

    distances = np.full((len(gt_boxes), len(pred_boxes)), np.nan, dtype=float)
    for i, gt in enumerate(gt_boxes):
        gx1, gy1, gx2, gy2 = gt
        gt_area = max(0.0, gx2 - gx1) * max(0.0, gy2 - gy1)
        for j, pred in enumerate(pred_boxes):
            px1, py1, px2, py2 = pred
            pred_area = max(0.0, px2 - px1) * max(0.0, py2 - py1)
            ix1 = max(gx1, px1)
            iy1 = max(gy1, py1)
            ix2 = min(gx2, px2)
            iy2 = min(gy2, py2)
            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
            union = gt_area + pred_area - inter
            iou = inter / union if union > 0 else 0.0
            if iou >= iou_threshold:
                distances[i, j] = 1.0 - iou
    return distances


def compute_panelc_tracking_metrics():
    acc = mm.MOTAccumulator(auto_id=True)
    frame_records = []

    for video in VIDEO_FOLDERS:
        images = list_images(video)
        for image_path in images:
            image_size = Image.open(image_path).size
            gt_path = GT_LABEL_ROOT / video / f'{image_path.stem}.txt'
            pred_path = PRED_LABEL_ROOT / video / f'{image_path.stem}.txt'

            gt_ids, gt_boxes = read_yolo_frame(gt_path, image_size, video)
            pred_ids, pred_boxes = read_yolo_frame(pred_path, image_size, video)
            distances = bbox_iou_matrix(gt_boxes, pred_boxes, IOU_THRESHOLD)
            acc.update(gt_ids, pred_ids, distances)

            frame_records.append({
                'video': video,
                'frame': image_path.stem,
                'gt_count': len(gt_ids),
                'pred_count': len(pred_ids),
            })

    metrics = [
        'num_frames',
        'num_objects',
        'num_unique_objects',
        'mota',
        'idf1',
        'idp',
        'idr',
        'precision',
        'recall',
        'num_misses',
        'num_false_positives',
        'num_switches',
        'num_fragmentations',
        'mostly_tracked',
        'partially_tracked',
        'mostly_lost',
    ]
    mh = mm.metrics.create()
    summary = mh.compute(acc, metrics=metrics, name=METHOD_NAME)
    frame_table = pd.DataFrame(frame_records)
    return summary, frame_table


In [ ]:
summary, frame_table = compute_panelc_tracking_metrics()
summary_path = OUTPUT_DIR / 'migfinder_tracking_summary.csv'
frame_table_path = OUTPUT_DIR / 'migfinder_frame_counts.csv'
summary.to_csv(summary_path)
frame_table.to_csv(frame_table_path, index=False)

score_metrics = {
    'MOTA': 'mota',
    'IDF1': 'idf1',
    'IDP': 'idp',
    'IDR': 'idr',
    #'Precision': 'precision',
    #'Recall': 'recall',
}
track_metrics = {
    'MT%': 'mostly_tracked',
    'PT%': 'partially_tracked',
    'ML%': 'mostly_lost',
}
count_metrics = {
    'FP': 'num_false_positives',
    'FN': 'num_misses',
    'IDs': 'num_switches',
    'FM': 'num_fragmentations',
}

summary_row = summary.loc[METHOD_NAME]
gt_track_count = float(summary_row['num_unique_objects'])
results = {metric: float(summary_row[column]) * 100.0 for metric, column in score_metrics.items()}
results.update({
    metric: (float(summary_row[column]) / gt_track_count) * 100.0 if gt_track_count > 0 else np.nan
    for metric, column in track_metrics.items()
})
results.update({metric: int(summary_row[column]) for metric, column in count_metrics.items()})
results_table = pd.DataFrame([results], index=[METHOD_NAME])
results_table.to_csv(OUTPUT_DIR / 'migfinder_tracking_metrics.csv')

print(summary)
print('\nSelected metrics:')
display(results_table)
print(f'Saved summary to: {summary_path}')
print(f'Saved frame counts to: {frame_table_path}')


In [ ]:
def add_bar_labels(ax, bars, fmt, errors=None):
    errors = np.zeros(len(bars)) if errors is None else np.asarray(errors)
    for bar, error in zip(bars, errors):
        height = bar.get_height()
        ax.annotate(
            fmt.format(height),
            xy=(bar.get_x() + bar.get_width() / 2, height + error),
            xytext=(0, 4),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=11,
        )


def compute_single_video_tracking_summary(video: str):
    acc = mm.MOTAccumulator(auto_id=True)
    for image_path in list_images(video):
        image_size = Image.open(image_path).size
        gt_path = GT_LABEL_ROOT / video / f'{image_path.stem}.txt'
        pred_path = PRED_LABEL_ROOT / video / f'{image_path.stem}.txt'
        gt_ids, gt_boxes = read_yolo_frame(gt_path, image_size, video)
        pred_ids, pred_boxes = read_yolo_frame(pred_path, image_size, video)
        distances = bbox_iou_matrix(gt_boxes, pred_boxes, IOU_THRESHOLD)
        acc.update(gt_ids, pred_ids, distances)

    metrics = [
        'num_frames', 'num_objects', 'num_unique_objects',
        'mota', 'idf1', 'idp', 'idr',
        'precision', 'recall', 'num_misses', 'num_false_positives',
        'num_switches', 'num_fragmentations', 'mostly_tracked',
        'partially_tracked', 'mostly_lost',
    ]
    mh = mm.metrics.create()
    return mh.compute(acc, metrics=metrics, name=video)


def compute_per_video_results():
    rows = []
    for video in VIDEO_FOLDERS:
        video_summary = compute_single_video_tracking_summary(video)
        row = video_summary.loc[video]
        gt_box_count = float(row['num_objects'])
        gt_track_count = float(row['num_unique_objects'])
        result = {'video': video, 'GT boxes': gt_box_count, 'GT tracks': gt_track_count}
        result.update({metric: float(row[column]) * 100.0 for metric, column in score_metrics.items()})
        for metric, column in track_metrics.items():
            result[metric] = (float(row[column]) / gt_track_count) * 100.0 if gt_track_count > 0 else np.nan
        for metric, column in count_metrics.items():
            result[metric] = (float(row[column]) / gt_box_count) * 100.0 if gt_box_count > 0 else np.nan
        rows.append(result)

    table = pd.DataFrame(rows).set_index('video')
    table.to_csv(OUTPUT_DIR / 'migfinder_tracking_metrics_by_video.csv')
    return table


def metric_means_and_errors(per_video_results: pd.DataFrame, metric_names: list[str]):
    means = per_video_results[metric_names].mean(skipna=True)
    errors = per_video_results[metric_names].sem(skipna=True).fillna(0.0)
    return means, errors


def plot_score_metrics(per_video_results: pd.DataFrame):
    metric_names = list(score_metrics.keys()) + list(track_metrics.keys())
    means, errors = metric_means_and_errors(per_video_results, metric_names)

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    bars = ax.bar(
        metric_names,
        means.values,
        yerr=errors.values,
        capsize=4,
        color='#4C78A8',
        label=f'{METHOD_NAME} mean',
        error_kw={'elinewidth': 1.2, 'capthick': 1.2},
    )
    ax.set_xlabel('metric')
    ax.set_ylabel('score (%)')
    upper = max(100, float((means + errors).max()) * 1.15 if len(means) else 100)
    ax.set_ylim(0, upper)
    ax.set_title('MigFinder Tracking Score Metrics (mean +/- SEM)')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.legend(frameon=False)
    add_bar_labels(ax, bars, '{:.1f}', errors.values)
    fig.tight_layout()
    out_path = OUTPUT_DIR / 'migfinder_score_metrics.pdf'
    fig.savefig(out_path, format='pdf', bbox_inches='tight')
    return fig, out_path


def plot_error_per_100_gt_boxes(per_video_results: pd.DataFrame):
    metric_names = list(count_metrics.keys())
    means, errors = metric_means_and_errors(per_video_results, metric_names)

    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    bars = ax.bar(
        metric_names,
        means.values,
        yerr=errors.values,
        capsize=4,
        color='#F58518',
        label=f'{METHOD_NAME} mean',
        error_kw={'elinewidth': 1.2, 'capthick': 1.2},
    )
    ax.set_xlabel('metric')
    ax.set_ylabel('events per 100 GT boxes')
    upper = max(1, float((means + errors).max()) * 1.15 if len(means) else 1)
    ax.set_ylim(0, upper)
    ax.set_title('MigFinder Tracking Error per 100 GT Boxes (mean +/- SEM)')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.legend(frameon=False)
    add_bar_labels(ax, bars, '{:.2f}', errors.values)
    fig.tight_layout()
    out_path = OUTPUT_DIR / 'migfinder_error_per_100_gt_boxes.pdf'
    fig.savefig(out_path, format='pdf', bbox_inches='tight')
    return fig, out_path


per_video_results = compute_per_video_results()
display(per_video_results)

score_fig, score_pdf = plot_score_metrics(per_video_results)
error_fig, error_pdf = plot_error_per_100_gt_boxes(per_video_results)
print(f'Saved score metrics figure to: {score_pdf}')
print(f'Saved error_per_100_gt_boxes figure to: {error_pdf}')
plt.show()
